**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**02 | MATLAB Object Decoding and Numerical Signal Recovery**

![Original MATLAB decoding and method-specific scientific validation](../../images/02_consolidated_workflow.svg)

**Research question:** How do we recover trustworthy numeric IMU and FE data from the original Essex MATLAB objects, and what is actually demonstrated by bounded SciPy inspection, a MATLAB-native converter contract and a pinned Python MATLAB table reader?

**Study input:** Original Essex Figshare article 30225739, v13, with 40 original participant files and 1,897 source-linked movement records. Source identity and trial provenance have been established in [Notebook 01](01_original_dataset_audit_and_trial_registry.ipynb). This notebook reads the original executed trial registry and SHA256 record in `output/` without overwriting them.

**Integrated methods:** Original MATLAB container and numeric-leaf audit (historical 03), optionally validated MATLAB-native conversion (historical 04), and a four-participant labelled Python `mat-io` pilot with original column and time-index inspection (historical 05).

**Historical evidence and interpretation:** The earlier Python pilot selected 28 movement records and validated 28 numerical IMU tables, but no FE numeric table passed that strict pilot. The MATLAB-native export was not supplied. This **does not mean numerical FE recovery is impossible**: subsequent full-cohort research (historical Stage 06) found 1,729 numerical FE records via a different audit path. Reader failures, numerical target semantics, timebase alignment and model readiness are distinct questions.

**Scientific protection:** The original executed Stages 03, 04 and 05 and all their CSV/PNG/JSON evidence remain unchanged. This integrated, unexecuted notebook saves only under `output/_consolidation_drafts/02_matlab_decoding/`, does not generate synthetic strain curves and does not train a prediction model. The optional full-source export is **not run automatically**.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Environment**

Standardise imports, source locations and experiment-specific output paths. Historical Stage 03–05 evidence is always read-only.


In [ ]:
# ======================== 01. Imports and Environment ========================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

# ---------------- 2. Standard Library and Python Environment -----------------
import json
import sys
from pathlib import Path
from importlib.metadata import version
from zipfile import ZIP_DEFLATED, ZipFile

# --------------- 3. Numerical Data and Original MAT Inspection ---------------
import numpy as np
import pandas as pd
from scipy.io import whosmat
from scipy.io.matlab import matfile_version

# --------------------- 4. Plotting and Notebook Display ----------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4))

# ------------- 5. Original Source, MATLAB and Decoder Utilities --------------
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID, ARTICLE_VERSION, fetch_version_manifest,
    find_published_file, ensure_source_file, inspect_matlab_archive,
    extract_matlab_files,
)
from essex_numeric_recovery import (
    check_record_registry, select_diagnostic_records,
    inspect_sampled_records, candidate_inventory,
    activity_fe_coverage, assert_source_fe_status_matches,
)
from essex_native_conversion import (
    EXPORT_COLUMNS, find_native_archive, sha256_file as native_sha256,
    unpack_native_archive, validate_native_manifest,
    validate_numeric_payloads, conversion_summary,
)
from essex_python_first import (
    PILOT_SUBJECTS, select_representative_rows, probe_pilot,
    evaluate_pilot, pilot_allows_expansion, inspect_native_table_schema,
)

# ----------- 6. Preserve Original Output and Isolate New Evidence ------------
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
historical_output_dir = study_dir / "output"
output_dir = historical_output_dir / "_consolidation_drafts" / "02_matlab_decoding"
export_dir = raw_dir / "native_export_v13"
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
assert output_dir != historical_output_dir
print("Original Figshare version:", ARTICLE_ID, ARTICLE_VERSION)
print("Consolidated source-decoding evidence:", output_dir)


**02. Approved Dataset and Trial Registry**

Check the verified archive, all 40 original MAT filenames and 1,897 source records against the original approved Stage 02 JSON and registry, without creating duplicate identities or resaving historic evidence.


In [ ]:
# ================== 02. Approved Dataset and Trial Registry ==================

# ------------- 1. Read the Frozen Trial Provenance and Registry --------------
provenance_path = historical_output_dir / "02_record_provenance_and_quality_gate.json"
registry_path = historical_output_dir / "02_original_record_registry.csv"
if not (provenance_path.is_file() and registry_path.is_file()):
    raise FileNotFoundError("Original executed dataset provenance is required.")
source_gate = json.loads(provenance_path.read_text(encoding="utf-8"))
assert source_gate["source_article_id"] == ARTICLE_ID
assert source_gate["version"] == ARTICLE_VERSION
assert source_gate["source_registry_gate"] == "PASSED"
registry = pd.read_csv(registry_path, dtype={
    "subject_id":"string", "source_file":"string",
    "local_record_key":"string", "task":"string",
    "updated_trial_id":"string", "fe_representation":"string",
    "imu_representation":"string",
})
identity_summary = check_record_registry(registry, source_gate)
assert_source_fe_status_matches(registry, source_gate)
assert identity_summary["records"] == 1897
assert identity_summary["participants"] == 40
assert registry["local_record_key"].is_unique

# ----------- 2. Verify the Published Archive and Actual MAT Files ------------
article, published_files = fetch_version_manifest()
archive_info = find_published_file(published_files, "MATLAB.zip")
zip_path, zip_hashes = ensure_source_file(archive_info, raw_dir)
assert zip_hashes["sha256"] == source_gate["archive_sha256"]
inventory = inspect_matlab_archive(zip_path, expected_mat_files=40)
mat_paths = extract_matlab_files(zip_path, mat_dir, inventory)
assert len(mat_paths) == 40
assert {p.stem.lower() for p in mat_paths} == set(registry["subject_id"])
pd.DataFrame([identity_summary]).to_csv(
    output_dir / "02_identity_summary.csv", index=False
)
print("Original participant and source-trial identity: VERIFIED")


**03. Source FE Coverage and MATLAB Containers**

Report task-level and participant-level source FE missingness, inspect MATLAB container variables and record file-level structural evidence. A source-empty field and a nonnumeric object representation must remain distinct.


In [ ]:
# =============== 03. Source FE Coverage and MATLAB Containers ================

# -------------- 1. Summarise Original Activity and FE Coverage ---------------
activity_counts = (
    registry.groupby("task").size().rename("record_count")
    .reset_index().sort_values("record_count", ascending=False)
)
fe_coverage = activity_fe_coverage(registry)
fe_table = fe_coverage.pivot(
    index="task", columns="fe_representation", values="record_count"
).fillna(0).astype(int)

display(activity_counts)
display(fe_table)

# --------------- 2. Audit Partial and Participant-Wide FE Gaps ---------------
subject_coverage = (
    registry.groupby(["subject_id", "fe_representation"])
    .size().rename("record_count").reset_index()
)
fe_coverage.to_csv(output_dir / "02_bounded_activity_fe_coverage.csv", index=False)
subject_coverage.to_csv(
    output_dir / "02_bounded_participant_fe_coverage.csv", index=False
)
print("Original activities:", registry["task"].nunique())
print("FE fields missing by representation:",
      int((registry["fe_representation"] == "empty").sum()))

# -------------- 3. Inspect Representative Original MAT Headers ---------------
container_rows = []
for source in mat_paths:
    major, minor = matfile_version(source)
    for variable_name, shape, matlab_class in whosmat(source):
        container_rows.append({
            "source_file": source.name,
            "mat_major": int(major),
            "mat_minor": int(minor),
            "source_variable": variable_name,
            "source_shape": str(tuple(shape)),
            "matlab_class": matlab_class,
        })
containers = pd.DataFrame(container_rows)
display(containers.head(12))
containers.to_csv(
    output_dir / "02_bounded_matlab_container_inventory.csv", index=False
)
print("MAT container variables recorded:", len(containers))
print("Files with MATLAB workspace reference:",
      containers.loc[
          containers["source_variable"] == "__function_workspace__",
          "source_file"
      ].nunique())


**04. Bounded SciPy Object and Numeric Leaf Audit**

Use a reproducible, bounded inspection of original MATLAB objects to preserve nested paths, numeric leaf candidates and the limits of shape-based inference. Do not treat candidate leaves as prediction-ready FE values.


In [ ]:
# ============== 04. Bounded SciPy Object and Numeric Leaf Audit ==============

# ----------- 1. Select Representative Published Record Identifiers -----------
selected_records = select_diagnostic_records(
    registry, max_samples=64
)
display(selected_records[[
    "subject_id", "task", "updated_trial_id",
    "local_record_key", "fe_representation"
]].head(16))
print("Source records selected for bounded introspection:",
      len(selected_records))

# --------- 2. Inspect Nested MATLAB Values without Implicit Decoding ---------
payload_paths, probe_summary = inspect_sampled_records(
    mat_dir, selected_records, max_depth=9, max_nodes=300
)
assert len(probe_summary) == len(selected_records) * 2

payload_paths.to_csv(
    output_dir / "02_bounded_matlab_object_path_inventory.csv", index=False
)
probe_summary.to_csv(
    output_dir / "02_bounded_probe_registry.csv", index=False
)
display(probe_summary.groupby("modality")[
    ["nodes_inspected", "numeric_leaves_visible", "inspection_truncated"]
].sum())
print("Truncated source-object probes:",
      int(probe_summary["inspection_truncated"].sum()))

# ----------- 3. Preserve All Numeric Leaf Candidates as Unverified -----------
candidate_table = candidate_inventory(payload_paths)
candidate_table.to_csv(
    output_dir / "02_bounded_numeric_leaf_candidates.csv", index=False
)
print("Numeric leaves discovered:", len(candidate_table))
print("Shape-compatible with 101 or 11 positions:",
      int(candidate_table["time_axis_candidate"].sum()) if len(candidate_table) else 0)

if candidate_table.empty:
    print("No attributable numerical IMU/FE arrays were recovered in sampled objects.")
else:
    display(candidate_table[[
        "subject_id", "modality", "path", "shape", "dtype",
        "beneath_matlab_object", "time_axis_candidate",
        "usable_for_prediction",
    ]].head(25))

# --------- 4. Record Actual Candidate Shapes without Assuming Units ----------
time_candidates = candidate_table.loc[
    candidate_table["time_axis_candidate"]
].copy()
time_candidates.to_csv(
    output_dir / "02_bounded_temporal_axis_candidates.csv", index=False
)
print("Target axis correspondence to published FE computation: UNVERIFIED")
print("Original MATLAB sensor / target column names: NOT YET VALIDATED")
print("Prediction-ready numeric tensor constructed: NO")


**05. Native Conversion Contract and Export Availability**

Define expected IMU and FE numerical shapes and check the optional MATLAB-native conversion route. If no original export is available, document AWAITING_NATIVE_EXPORT rather than inventing data.


In [ ]:
# ========== 05. Native Conversion Contract and Export Availability ===========

# ------------- 1. Summarise Original FE Availability by Activity -------------
original_fe = (
    registry.groupby(["task", "fe_representation"])
    .size().rename("records").reset_index()
)
assert original_fe["records"].sum() == 1897
assert registry["fe_representation"].eq("empty").sum() == 168
display(original_fe)

# ------------ 2. Record Required Numerical and Scientific Checks -------------
contract = pd.DataFrame([
    {"modality":"IMU", "records":1897, "published_channels":42,
     "published_time_grid":"101 normalised points",
     "unit_and_axis_verification":"pending"},
    {"modality":"FE", "records":1897, "published_channels":2,
     "published_time_grid":"every 10th position; verify original grid",
     "unit_and_axis_verification":"pending"},
])
contract.to_csv(output_dir / "02_native_expected_conversion_contract.csv",index=False)
display(contract)

# ------------ 3. Inspect the Version-Controlled Native Converter -------------
matlab_source = study_dir / "code" / "matlab" / "export_essex_native_tables.m"
if matlab_source.is_file():
    print("MATLAB-native converter available in local repository:", matlab_source.name)
else:
    # Conversion runs in MATLAB Online/Desktop, not in this Colab runtime.
    print("MATLAB-native converter not found in this local checkout.")
    print("It is available in the current GitHub repository;")
    print("the Python pre-conversion audit can proceed without it.")

# ---------- 4. Resolve a Single Optional Converted Numeric Archive -----------
native_archive = find_native_archive(raw_dir)
has_export = native_archive is not None
print("MATLAB-native ZIP supplied:", has_export)
if has_export:
    print("Source conversion archive SHA256:", sha256_file(native_archive))
else:
    print("Conversion status: AWAITING_NATIVE_EXPORT")
    print("No numeric source signals will be invented.")


**06. Native Manifest, Payload and Time-Grid Checks**

If a real conversion archive exists, validate original record identities, finite numerical payloads and candidate shapes. Retain unknown physical units and unresolved target time correspondence explicitly.


In [ ]:
# ============= 06. Native Manifest, Payload and Time-Grid Checks =============

# ---------- 1. Recover Approved Numeric Export Metadata If Supplied ----------
if has_export:
    validated_dir = unpack_native_archive(native_archive, export_dir)
    native_manifest = validate_native_manifest(
        validated_dir / "export_manifest.csv", registry
    )
    print("Native manifest source identities: VERIFIED")
else:
    validated_dir = None
    native_manifest = pd.DataFrame(
        columns=[*EXPORT_COLUMNS,"local_record_key"]
    )
native_manifest.to_csv(
    output_dir / "02_native_conversion_manifest.csv",index=False
)

# ---------------- 2. Describe the Observed Conversion Status -----------------
if has_export:
    conversion_status = (
        native_manifest.groupby(["modality","status"])
        .size().rename("records").reset_index()
    )
else:
    conversion_status = pd.DataFrame([
        {"modality":"IMU","status":"awaiting_native_export","records":1897},
        {"modality":"FE","status":"awaiting_native_export","records":1897},
    ])
conversion_status.to_csv(
    output_dir / "02_native_conversion_status_summary.csv",index=False
)
display(conversion_status)

# ------------- 3. Read All Real Numeric Payloads When Available --------------
payload_cols = [
    "local_record_key","subject_id","task","updated_trial_id",
    "modality","conversion_status","n_time","n_channels",
    "finite","label_count_matches","time_101_candidate",
    "fe_11_candidate","expected_imu_42_channels",
    "named_fe_e1_e3_candidate","sha256"
]
if has_export:
    payload_audit = validate_numeric_payloads(
        validated_dir,native_manifest
    )
    native_summary = conversion_summary(native_manifest,payload_audit)
    assert len(payload_audit)==len(native_manifest)
else:
    payload_audit = pd.DataFrame(columns=payload_cols)
    native_summary = {
        "manifest_rows":0, "exported_numeric_payloads":0,
        "validated_finite_payloads":0, "imu_time_101_candidate":0,
        "imu_42_channel_candidate":0, "fe_time_11_candidate":0,
        "fe_named_pair_candidate":0, "conversion_status_counts":{},
        "physical_units_verified":False,
        "imu_fe_temporal_alignment_verified":False,"model_ready":False
    }
payload_audit.to_csv(
    output_dir/"02_native_numeric_payload_validation.csv",index=False
)
display(pd.DataFrame([{k:v for k,v in native_summary.items()
                      if k!="conversion_status_counts"}]))

# ----------- 4. Enforce Separation of Numeric and Model Readiness ------------
assert not native_summary["model_ready"]
assert not native_summary["physical_units_verified"]
assert not native_summary["imu_fe_temporal_alignment_verified"]
print("Finite numerical arrays validated:",
      native_summary["validated_finite_payloads"])
print("Biomechanical and FE temporal semantics: NOT VERIFIED")

# ------------ 5. Record Numeric Shape and Column-Name Candidates -------------
if has_export:
    finite_rows = payload_audit.loc[
        payload_audit["finite"] == True
    ].copy()
else:
    finite_rows = pd.DataFrame(columns=payload_cols)
finite_rows.to_csv(output_dir/"02_native_numeric_grid_candidates.csv",index=False)
time_audit = pd.DataFrame([
    {"criterion":"IMU sequence with 101 source positions",
     "candidates":native_summary["imu_time_101_candidate"],"verified":False},
    {"criterion":"IMU sequence with 42 source channels",
     "candidates":native_summary["imu_42_channel_candidate"],"verified":False},
    {"criterion":"FE sequence with 11 source positions",
     "candidates":native_summary["fe_time_11_candidate"],"verified":False},
    {"criterion":"FE channels explicitly named e1 and e3",
     "candidates":native_summary["fe_named_pair_candidate"],"verified":False},
])
display(time_audit)
time_audit.to_csv(output_dir/"02_native_temporal_semantics_audit.csv",index=False)

# ------------- 6. Preserve the Unresolved Scientific Boundaries --------------
print("IMU channel units and anatomical axes: UNVERIFIED")
print("FE e1/e3 sign and physical units: UNVERIFIED")
print("Native FE sampling and paired timebase: UNVERIFIED")
print("No FE interpolation, scalar imputation or model training performed.")


**07. Pinned Python Table Decoder**

Install the specifically tested original MATLAB table reader and verify its version. The source identities and output labels must remain unchanged through decoding.


In [ ]:
# ====================== 07. Pinned Python Table Decoder ======================

# -------------- 1. Install the Historical Pinned MATLAB Reader ---------------
%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat

# ------------------ 2. Verify the Installed Package Version ------------------
assert version("mat-io") == "1.0.1"
print("Python reader:", version("mat-io"))
print("Python runtime:", sys.version.split()[0])


**08. Source-Linked Pilot Selection and Container Preflight**

Select the authentic four-participant pilot from approved original trial IDs; inspect each source MAT container, retaining full file and trial traceability. Use a bounded sample instead of repeatedly decoding the complete archive.


In [ ]:
# ========= 08. Source-Linked Pilot Selection and Container Preflight =========

registry_summary = check_record_registry(registry, source_gate)
assert registry_summary["records"] == 1897
assert registry_summary["participants"] == 40
assert {x.stem.lower() for x in mat_paths} == set(registry["subject_id"])
assert int(registry["fe_representation"].eq("empty").sum()) == 168

selected = select_representative_rows(
    registry, subjects=PILOT_SUBJECTS, max_each=8
)
selected.to_csv(
    output_dir / "02_python_pilot_selection.csv", index=False
)
display(selected[[
    "subject_id", "task", "updated_trial_id",
    "local_record_key", "fe_representation",
]])
print("Original pilot records:", len(selected))

preflight_rows = []
for subject in PILOT_SUBJECTS:
    path = mat_dir / f"{subject}.mat"
    for variable, shape, matlab_class in whosmat(path):
        preflight_rows.append({
            "subject_id": subject, "source_file": path.name,
            "variable": variable, "shape": str(tuple(shape)),
            "matlab_class": matlab_class,
        })
preflight = pd.DataFrame(preflight_rows)
for subject in PILOT_SUBJECTS:
    assert "Data" in set(
        preflight.loc[preflight["subject_id"] == subject, "variable"]
    )
preflight.to_csv(
    output_dir / "02_python_source_preflight.csv", index=False
)
display(preflight)


**09. Python IMU and FE Decoding with Column Schema**

Decode original labelled IMU and FE MATLAB objects from the pilot, record reader errors, and retain original column names and time-index metadata. Descriptive column types are not in themselves proof of validated numerical target matrices.


In [ ]:
# ============= 09. Python IMU and FE Decoding with Column Schema =============

pilot, errors, decoded = probe_pilot(
    load_from_mat, mat_dir, selected
)
assert len(pilot) == 2 * len(selected)
pilot.to_csv(
    output_dir / "02_python_decode_audit.csv", index=False
)
errors.to_csv(
    output_dir / "02_python_reader_errors.csv", index=False
)
print("Original record-modality probes:", len(pilot))
print("File-level decoding errors:", len(errors))
print("Original-column-labelled numeric candidates:", len(decoded))
display(pilot[[
    "source_file", "local_record_index", "modality",
    "decode_status", "source_identity_match",
    "python_class", "shape", "n_time", "n_channels",
]].head(30))
if not errors.empty:
    display(errors)

column_schema, index_schema = inspect_native_table_schema(
    load_from_mat, mat_dir, selected
)
column_schema.to_csv(
    output_dir / "02_python_column_schema.csv", index=False
)
index_schema.to_csv(
    output_dir / "02_python_index_schema.csv", index=False
)
column_summary = (
    column_schema.groupby(
        ["modality", "original_column_name", "pandas_dtype",
         "is_numeric_dtype", "is_time_label_candidate"],
        dropna=False,
    ).size().rename("source_records").reset_index()
)
print("Original table-column metadata rows:", len(column_schema))
print("Source-index metadata records:", len(index_schema))
display(column_summary.head(100))
display(index_schema.groupby(
    ["modality", "table_class", "n_rows", "n_columns", "index_class"],
    dropna=False,
).size().rename("source_records").reset_index())
print("No original column excluded or relabelled.")


**10. Strict Decoder Results and Export Decision**

Compare verified IMU and FE numerical-table counts, original source labels and reader errors. Preserve the strict method-specific gate without automatically running an expensive full-source export.


In [ ]:
# ============== 10. Strict Decoder Results and Export Decision ===============

pilot_results = evaluate_pilot(pilot, errors)
allow_full = pilot_allows_expansion(pilot, errors)

status = (
    pilot.groupby(["modality", "decode_status"])
    .size().rename("count").reset_index()
)
status.to_csv(
    output_dir / "02_python_decoding_status.csv", index=False
)
candidates = pilot.loc[
    pilot["decode_status"] == "labelled_numeric_table"
].copy()
candidates.to_csv(
    output_dir / "02_python_labelled_candidates.csv", index=False
)
display(status)
print("Reader-error files:", pilot_results["reader_error_files"])
print("Verified original record identities:",
      pilot_results["identity_matches"])
print("Finite named IMU tables:", pilot_results["labelled_numeric_imu"])
print("Finite named FE tables:", pilot_results["labelled_numeric_fe"])
print("Candidate 101 x 42 IMU tables:",
      pilot_results["imu_101_42_pairs"])
print("Candidate FE 11-point e1/e3 tables:",
      pilot_results["fe_11_e1e3_pairs"])
print("Full-source expansion:", "PASSED" if allow_full else "BLOCKED")

# ------------ 2. Preserve the Strict Full-Source Export Boundary -------------
RUN_FULL_EXPORT = False
export_state = (
    "ELIGIBLE_REQUIRES_APPROVAL" if allow_full
    else "BLOCKED_BY_STRICT_PILOT"
)
export_status = pd.DataFrame([{
    "strict_pilot_passed": bool(allow_full),
    "automatic_full_source_export_performed": RUN_FULL_EXPORT,
    "export_decision": export_state,
}])
export_status.to_csv(
    output_dir / "02_python_full_export_decision.csv", index=False
)
print("Full-source export:", export_state)
print("No new source-wide numeric export was produced.")


**11. Original Decoder Evidence and Real IMU Plot**

Plot authentic pilot reader classifications and a real decoded IMU column, without generating unsupported FE curves or repeating the source-level distribution figure from Notebook 01.


In [ ]:
# ============== 11. Original Decoder Evidence and Real IMU Plot ==============

fig, ax = plt.subplots(
    1, 2, figsize=(13.5, 5),
    gridspec_kw={"width_ratios": [1.65, 1.0]},
    constrained_layout=True,
)
by_status = status.pivot(
    index="decode_status", columns="modality", values="count"
).fillna(0).astype(int).sort_index()
x = np.arange(len(by_status))
w = 0.34
for off, label in [(-0.5, "IMU"), (0.5, "FE")]:
    values = (
        by_status[label].to_numpy()
        if label in by_status else np.zeros(len(by_status))
    )
    ax[0].bar(x + off * w, values, w, label=label)
ax[0].set_xticks(x)
ax[0].set_xticklabels(
    [v.replace("_", " ") for v in by_status.index],
    rotation=25, ha="right"
)
ax[0].set_title("Original Python MATLAB object decoding outcomes")
ax[0].set_ylabel("Original pilot record-modality entries")
ax[0].legend(frameon=False)
ax[0].grid(axis="y", alpha=0.2)

imu_keys = [k for k in decoded if k[1] == "IMU"]
if imu_keys:
    real_values, original_names = decoded[imu_keys[0]]
    ax[1].plot(
        np.arange(real_values.shape[0]),
        real_values[:, 0], linewidth=1.5
    )
    ax[1].set_title("Original numeric IMU column (units unverified)")
    ax[1].set_xlabel("Source sample index")
    ax[1].set_ylabel("Numerical value")
else:
    ax[1].text(
        0.5, 0.5, "No source-labelled numerical IMU waveform",
        ha="center", va="center", transform=ax[1].transAxes
    )
    ax[1].set_title("Authentic numerical waveform availability")
    ax[1].set_xticks([])
    ax[1].set_yticks([])
save_figure(fig, output_dir / "02_python_decoder_diagnostics.png")
plt.show()


**12. Integrated Decoder Evidence and Quality Gate**

Reconcile the previous SciPy, MATLAB-native and Python-first Quality Gates. A passing consistency audit establishes traceability, not end-to-end FE readiness or the failure of later recovery methods.


In [ ]:
# ============= 12. Integrated Decoder Evidence and Quality Gate ==============

# ---------- 1. Verify the Source, Object Audit and Pilot Accounting ----------
assert source_gate["source_article_id"] == ARTICLE_ID
assert zip_hashes["sha256"] == source_gate["archive_sha256"]
assert len(mat_paths) == 40 and len(registry) == 1897
assert len(probe_summary) == 2 * len(selected_records)
assert probe_summary["local_record_key"].nunique() == len(selected_records)
assert len(pilot) == 2 * len(selected)
assert len(errors) == pilot_results["reader_error_files"]
assert pilot_results["identity_matches"] == len(selected)
assert registry["local_record_key"].is_unique
assert not candidate_table.get(
    "usable_for_prediction", pd.Series([], dtype=bool)
).any()

# ------- 2. Read Historical Method Quality Gates Without Modification --------
historical_quality_files = [
    "03_numeric_recovery_quality_gate.json",
    "04_native_conversion_quality_gate.json",
    "05_python_first_quality_gate.json",
]
history = {}
for filename in historical_quality_files:
    path = historical_output_dir / filename
    if not path.is_file():
        raise FileNotFoundError("Missing prior method evidence: " + filename)
    history[filename] = json.loads(path.read_text(encoding="utf-8"))
assert history[historical_quality_files[0]]["archive_sha256"] == zip_hashes["sha256"]
assert history[historical_quality_files[1]]["original_matlab_zip_sha256"] == zip_hashes["sha256"]
assert history[historical_quality_files[2]]["source_sha256"] == zip_hashes["sha256"]

# ------- 3. Save Integrated Decoder and Scientific Readiness Evidence --------
quality = {
    "figshare_article": ARTICLE_ID,
    "version": ARTICLE_VERSION,
    "source_sha256": zip_hashes["sha256"],
    "source_records": len(registry),
    "participants": len(mat_paths),
    "bounded_matlab_records": len(selected_records),
    "bounded_modality_probes": len(probe_summary),
    "numeric_leaf_candidates_unverified": len(candidate_table),
    "native_export_supplied": bool(has_export),
    "native_validated_finite_payloads": int(
        native_summary["validated_finite_payloads"]
    ),
    "python_reader": "mat-io",
    "python_reader_version": version("mat-io"),
    "python_pilot_records": len(selected),
    "python_strict_numeric_imu": int(pilot_results["labelled_numeric_imu"]),
    "python_strict_numeric_fe": int(pilot_results["labelled_numeric_fe"]),
    "python_reader_error_files": len(errors),
    "strict_expansion_gate": bool(allow_full),
    "automatic_full_source_export_performed": False,
    "historical_method_evidence_read_only": historical_quality_files,
    "decoding_consistency_gate": "PASSED",
    "physical_units_verified_here": False,
    "imu_fe_alignment_verified_here": False,
    "regression_model_ready_here": False,
    "model_training_performed": False,
}
(output_dir / "02_consolidated_decoding_quality_gate.json").write_text(
    json.dumps(quality, indent=2) + "\n", encoding="utf-8"
)
print("CONSOLIDATED 02 | SCIENTIFIC DECODING AUDIT: PASSED")
print("Method-specific pilot permits full export:", bool(allow_full))
print("No synthetic targets or model training performed.")


**13. Colab Output Validation and Safe Download**

Validate every saved CSV, PNG and JSON generated by this notebook; use the mandatory standard Stage 02 downloader plus a separate ZIP containing only this version's results. Historical 02/03/04/05 results are never overwritten.


In [ ]:
# =============== 13. Colab Output Validation and Safe Download ===============

# --------------- 1. Check the Complete New Evidence Inventory ----------------
expected = [
    "02_record_provenance_and_quality_gate.json",
    "02_original_record_registry.csv",
    "02_identity_summary.csv",
    "02_bounded_activity_fe_coverage.csv",
    "02_bounded_participant_fe_coverage.csv",
    "02_bounded_matlab_container_inventory.csv",
    "02_bounded_matlab_object_path_inventory.csv",
    "02_bounded_probe_registry.csv",
    "02_bounded_numeric_leaf_candidates.csv",
    "02_bounded_temporal_axis_candidates.csv",
    "02_native_expected_conversion_contract.csv",
    "02_native_conversion_manifest.csv",
    "02_native_conversion_status_summary.csv",
    "02_native_numeric_payload_validation.csv",
    "02_native_numeric_grid_candidates.csv",
    "02_native_temporal_semantics_audit.csv",
    "02_python_pilot_selection.csv",
    "02_python_source_preflight.csv",
    "02_python_decode_audit.csv",
    "02_python_reader_errors.csv",
    "02_python_column_schema.csv",
    "02_python_index_schema.csv",
    "02_python_decoding_status.csv",
    "02_python_labelled_candidates.csv",
    "02_python_full_export_decision.csv",
    "02_python_decoder_diagnostics.png",
    "02_consolidated_decoding_quality_gate.json"
]
missing = [p for p in expected if not (output_dir / p).is_file()]
if missing:
    raise FileNotFoundError("Missing consolidated evidence: " + ", ".join(missing))
print("Consolidated decoding artefacts:", len(expected))

# ----------- 2. Use the Standard Colab Research-Output Downloader ------------
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="02"
)

# ----------- 3. Package Only the New Notebook's Scientific Results -----------
archive = Path("/content") / "BoneNet_Consolidated_02_Outputs.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as package:
    for name in expected:
        package.write(output_dir / name, arcname=name)
print("Consolidated-only archive:", archive)
print("Original research outputs changed: NO")
print("GitHub commit or push performed by notebook: NO")


**14. Scientific Interpretation and Next Research Boundary**

**Scientific contribution:** This notebook unifies the methodological investigations previously separated among original Stages 03, 04 and 05. It contrasts bounded nested-object inspection, an optionally validated MATLAB-native export contract and a pinned Python MATLAB table-reader pilot while keeping all results source-linked.

**Earlier observations, not newly executed conclusions:** The original Stage 03 numeric-object scan did not produce prediction-ready IMU/FE tables; Stage 04 reported AWAITING_NATIVE_EXPORT; Stage 05 recovered 28 numeric IMU tables from 28 source records, but no FE numeric table passed its strict initial pilot. These methods and their outcomes have distinct scopes. Historical Stage 06 later recovered 1,729 numerical FE records via full-cohort analysis, which must not be invalidated by this earlier method-specific pilot.

**Reproducibility:** Versioned Figshare article, original trial registry, strict reader version and SHA256 checks are retained. The completed old Stages 03–05 and their figures and data exports are preserved in Git history and the main output directory.

**Limitations:** No fabricated FE, no interpolation to 101 points, no physical unit claims, no unbiased participant-level Train/Test protocol, and no external or clinical validation. A consistency pass is not a model-readiness claim.

**Next distinct topic:** Once this consolidated notebook has executed and its saved evidence has been audited, the following consolidated notebook will integrate historical Stages 06–07: full-cohort numeric validation, original timestep matching, and accurate FE missingness reconciliation. Do not delete executed predecessor notebooks until this replacement is independently confirmed.
